Kaggle Dataset

In [ ]:
!kaggle datasets download -d syncoraai/credit-scoring-dataset
!unzip credit-scoring-dataset.zip

Load Data

In [ ]:
import pandas as pd

# Load the dataset
# (Make sure the filename matches exactly what you downloaded)
df = pd.read_csv('synthetic_e2dabba50a1a4fbcabd601f7883eef1e.csv')

# Let's peek at the first 5 rows to identify our Target column
display(df.head())

Knowing Colomn Target

In [ ]:
# Print all column names as a list
print(df.columns.tolist())

Data Audit

In [ ]:
# 1. [ALWAYS] Check Dataset Size
print(f"Dataset Size: {df.shape[0]} rows, {df.shape[1]} columns")

# 2. [ALWAYS] Check Feature Types
print("\nFeature Types:")
print(df.dtypes.value_counts())

# 3. [ALWAYS] Check for Duplicate Rows
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print("Action taken: Duplicates removed.")

# 4. [ALWAYS] Check Class Balance
# REPLACE 'Target' with the actual name of your label column!
target_col = 'CREDIT_SCORE'
print("\nClass Balance:")
print(df[target_col].value_counts(normalize=True) * 100)

Exploratory Data Analysis

In [ ]:
# 1. [ALWAYS] Missing Value Audit (Page 4)
missing_counts = X_train.isna().sum()
missing_cols = missing_counts[missing_counts > 0].sort_values(ascending=False)

print(f"\nColumns with missing values: {len(missing_cols)}")
if len(missing_cols) > 0:
    print(missing_cols.head(10)) # Show top 10 missing
else:
    print("Great! No missing values detected.")

# 2. Identify the 2 "Object" (Text/Categorical) columns you found earlier
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns to encode: {cat_cols}")

for col in cat_cols:
    print(f"\nUnique values in '{col}':")
    print(X_train[col].value_counts().head()) # Show top 5 categories

Stratified Split

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Create your custom target
df['Is_Bad_Credit'] = (df['CREDIT_SCORE'] < 600).astype(int)

# 2. DROP THE LEAKAGE (Page 22 Checklist)
# We drop CUST_ID (useless), CREDIT_SCORE (target source),
# and DEFAULT (future information)
X = df.drop(['CUST_ID', 'CREDIT_SCORE', 'DEFAULT', 'Is_Bad_Credit'], axis=1)
y = df['Is_Bad_Credit']

# 3. Apply the Stratified Split (The Golden Rule - Page 4)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42
)

print(f"Target: Is_Bad_Credit (Score < 600)")
print(f"Features in X_train: {X_train.shape[1]}")
print(f"Rows in X_train: {X_train.shape[0]}")

Build Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Automatically identify numeric vs categorical columns in X_train
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Found {len(numeric_features)} Numeric Features")
print(f"Found {len(categorical_features)} Categorical Features: {categorical_features}")

# 2. Create the Numeric Pipeline (Page 7 - StandardScaler for Logistic Reg/SVM)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fills missing numbers with median
    ('scaler', StandardScaler())                   # Standardizes to mean=0, variance=1
])

# 3. Create the Categorical Pipeline (Page 7 - One-Hot Encoding)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fills missing text with most common
    ('encoder', OneHotEncoder(handle_unknown='ignore'))   # Converts text to binary columns
])

# 4. Combine them using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("\nSuccess! Preprocessor built. It will handle all 85 columns automatically without data leakage.")

Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# 1. Create the full Pipeline (Preprocessor + Model)
# We use 100 trees as a baseline (Page 13)
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1 # Uses all your CPU cores for faster training
)

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

# 2. TRAIN the pipeline on X_train and y_train
# This fits the scaler, the encoder, AND the model all at once!
clf_pipeline.fit(X_train, y_train)

print("Training Complete!")

Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Predict on the Validation Set
y_pred = clf_pipeline.predict(X_val)

print("--- PHASE 11: EVALUATION (VALIDATION SET) ---")
print(classification_report(y_val, y_pred))

# Plot Confusion Matrix (Page 17 [ALWAYS])
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Random Forest: Credit Default Confusion Matrix")
plt.show()

Model Interpretability

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate importance on the Validation Set
result = permutation_importance(clf_pipeline, X_val, y_val, n_repeats=10, random_state=42)

# Get the top 10 features
import pandas as pd
feature_importances = pd.Series(result.importances_mean, index=X_train.columns)
print("\n--- TOP 10 PREDICTIVE FEATURES ---")
print(feature_importances.sort_values(ascending=False).head(10))

HYPERPARAMETER TUNING

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# 1. Define the parameters to test (Page 12 recommendations)
param_dist = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_leaf': [1, 2, 4, 8],
    'classifier__max_features': ['sqrt', 'log2']
}

# 2. Run the Search
# Note: We run this ON THE PIPELINE to keep the preprocessing leak-proof!
random_search = RandomizedSearchCV(
    clf_pipeline,
    param_distributions=param_dist,
    n_iter=10,           # Test 10 random combinations
    cv=3,                # 3-fold cross-validation
    scoring='f1',        # Optimize for F1-Score (Page 17)
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(f"Best Parameters Found: {random_search.best_params_}")
print(f"Best Cross-Validation F1-Score: {random_search.best_score_:.4f}")

Final Model Evaluation

In [ ]:
# 3. Use the best found model
best_model = random_search.best_estimator_
y_pred_tuned = best_model.predict(X_val)

print("\n--- TUNED MODEL PERFORMANCE ---")
print(classification_report(y_val, y_pred_tuned))

# Compare Confusion Matrices
cm_tuned = confusion_matrix(y_val, y_pred_tuned)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_tuned)
disp.plot(cmap='Greens')
plt.title("Tuned Random Forest: Confusion Matrix")
plt.show()